# The great plan that will make me write my thesis

- What kinds of plots do I have?
- Which ones can I potentially use?
- Which plots do I have to redo to use them?
- What story can I spin around this stuff?

## First experiments

Before starting in earnest, I made a few figures to develop an intuition for the effect of 
- different viscosities
- different sigma slopes -> almost no difference in the azimuthally averaged $\Sigma/\Sigma_0$ plot (What if we don't average?). unclear if coincidence or generally true fact. Potentially interesting!
- different aspect ratios -> lower aspect ratio leads to deeper gap, because there's less pressure to push the gas back into the gap. 
- different inner radius -> relevant for the formation of the inner bump!
- different boundary conditions -> has virtually no effect; irrelevant
- development over time of the simulation -> gap formation
- logarithmic grid -> not really interesting to show, was just relevant as sanity check.
- one multifluid sim with several kinds of dust, but with no relevant conclusions.
- I never explored different values for the flaring index.

This was all with the wrong planet mass (one jupiter mass)



## The radmc3d images
Interesting because of spiral/no spiral situation, see-through and not see-through gap, shadows that regions of higher pressure make that could explain the outer dark rings in WISPIT 2

Parameters:
- aspect ratio 0.05
- flaring index 0.0
- sigma slope 1.0
- alpha 1.0e-2, 1.0e-3, 1.0e-4
- sigma0 6.3661977237e-4 -> quite irrelevant, since radmc3d normalized it again. I think $\Sigma_{dust}$ was about 0.1 g/cm² at 20 au.
- Ymin			0.4 
- Ymax			2.5
- The planet I used was too light. It has one jupiter mass . . .

Quick estimate for mass of disk:

In [8]:
import numpy as np
au  = 1.49598e13     # Astronomical Unit       [cm]
ms  = 1.98892e33     # Solar mass              [g]

# quick and dirty estimate for disk mass under these conditions
sigslop=1
sigma_0 = 1e0 # g/cm² gas density at planet position (approximately)
M = 57**2*(2.5**(2-sigslop)-0.4**(2-sigslop))/(2-sigslop)*57**sigslop*au**2*sigma_0*2*np.pi #g/cm²
print(f"Estimate for disk mass: {M/ms} solar masses")

Estimate for disk mass: 0.27495270079873557 solar masses


That's not terrible, but it's not great, either. 
I think because the planet mass is incorrect I should redo it.

### Important question: If I redo it, what parameters should I choose? What should I do differently?

- logarithmic r grid.
- Grid should extend up to 300 au, so ~ 5 should be chosen as outer boundary radius.
- Inner radius should be 20 au (maybe?) so ~ 0.3
- Aspect ratio and flaring index should be cross-checked with reasonable values one can achieve with the star we have observed. Values like 0.08 aspect ratio at planet and a flaring index of 0.25 seem reasonable.
- planet mass should be chosen correctly! I have defined a new planet config file for the planet in WISPIT 2.
- SigmaSlope and the sigma0 we use later for radmc3d should be checked. Disk mass should be (significantly ?) lower than stellar mass. However, sigma0 can still be adapted later if necessary. Maybe also try a crazy valu for SigmaSlope, as suggested by the behaviour of the scattering surface?
- Check if radial resolution is sufficient . . .

In [4]:
# The cross check for flaring index and aspect ratio:
from model_check import model_check

model_check(sigma_slope = 0, sigma_0= 4)

INPUT:
sigma_0: 4 g/cm² 
sigma_slope : 0
flang: 0.05
r_in: 0.3r0
r_out: 6r0
r0 : 57au
------
STAR input
mstar: 1.08 solar masses
rstar: 1.418 r_sun
tstar: 4400K
-------
These are reasonable values for the aspect ratio and the flaring index, given the parameters:
  AspectRatio in model at R0  = 0.06912616677247987
  FlaringIndex in model at R0 = 0.24968493690647145
Mass of the disk: 0.1649716204792413 solar masses.


In [12]:
"""
Setup			fargo_alpha_visc
### Disk parameters

AspectRatio     	0.05            Thickness over Radius in the disc
Sigma0			6.3661977237e-4	Surface Density at r=1
Alpha 			1.0e-2
SigmaSlope		1.0		Slope for the surface density
FlaringIndex		0.0		Slope for the aspect-ratio

### Radial range for damping (in period-ratios). Values smaller than one
### prevent damping.

DampingZone 1.15

### Characteristic time for damping, in units of the inverse local
### orbital frequency. Higher values means lower damping

TauDamp 0.3

### Planet parameters

PlanetConfig		planets/jupiter.cfg
ThicknessSmoothing 	0.6
RocheSmoothing 		0.0
Eccentricity		0.0
ExcludeHill		no
IndirectTerm		Yes

### Mesh parameters

Nx			384		Azimuthal number of zones
Ny               	128		Radial number of zones
Xmin			-3.14159265358979323844	
Xmax			3.14159265358979323844
Ymin			0.4		Inner boundary radius
Ymax			2.5		Outer boundary radius
OmegaFrame     		1.0005		Angular velocity for the frame of reference (If Frame is F).
Frame			G		Method for moving the frame of reference

### Output control parameters

DT			0.314159265359	Physical time between fine-grain outputs
Ninterm	 		200		Number of DTs between scalar fields outputs
Ntot			20000		Total number of DTs



### Plotting parameters

PlotLog			yes
"""

'\nSetup\t\t\tfargo_alpha_visc\n### Disk parameters\n\nAspectRatio     \t0.05            Thickness over Radius in the disc\nSigma0\t\t\t6.3661977237e-4\tSurface Density at r=1\nAlpha \t\t\t1.0e-2\nSigmaSlope\t\t1.0\t\tSlope for the surface density\nFlaringIndex\t\t0.0\t\tSlope for the aspect-ratio\n\n### Radial range for damping (in period-ratios). Values smaller than one\n### prevent damping.\n\nDampingZone 1.15\n\n### Characteristic time for damping, in units of the inverse local\n### orbital frequency. Higher values means lower damping\n\nTauDamp 0.3\n\n### Planet parameters\n\nPlanetConfig\t\tplanets/jupiter.cfg\nThicknessSmoothing \t0.6\nRocheSmoothing \t\t0.0\nEccentricity\t\t0.0\nExcludeHill\t\tno\nIndirectTerm\t\tYes\n\n### Mesh parameters\n\nNx\t\t\t384\t\tAzimuthal number of zones\nNy               \t128\t\tRadial number of zones\nXmin\t\t\t-3.14159265358979323844\t\nXmax\t\t\t3.14159265358979323844\nYmin\t\t\t0.4\t\tInner boundary radius\nYmax\t\t\t2.5\t\tOuter boundary radi

## The fargo3d images

I also have a sequence of images from the fargo simulations performed for different viscosities.
That could be of interest, too. For example, it's interesting that the spirals in the hydro simulation are visible in the radcm3d scattered-light images.

## New Simulations with correct parameters

#### Is 1000 orbits enough? 
Fixed:

- flang: 0.05
- r_in: 0.3r0 (ca. 20 au)
- r_out: 7r0 (ca. 400 au)
- r0 : 57au
- AspectRatio in model at R0  = 0.06912616677247987
- FlaringIndex in model at R0 = 0.24968493690647145

STAR input
- mstar: 1.08 solar masses
- rstar: 1.418 r_sun
- tstar: 4400K



unclear:
- sigma_slope : 0.5,1, something else
- sigma_0: choose for radmc3d only, make sure disk mass is sensible
- planet mass: don't trust paper blindly, mass obtained with magic. I just checked and they compare the magnitudes in different bands with isochrones of different evolutionary models for the estimated stellar age. So there is a lot of room for errors. Maybe I could try something like 2, 5 and 10 jupiter masses? 5 should be approximtely right, and then we can also check the effect of a less or a more massive planet.
- alpha viscosity: 1e-2, 1e-3, 1e-4.


Masses for the planet input files:

In [5]:
from uncertainties import ufloat
jupiter_mass = ufloat(9.547919e-4,0.000002e-4) ## in solar masses

# There is no way to include asymmetric errors in uncertainties
# 4.9 is the mass in jupiter masses
Wispit_mass_upper = ufloat(4.9,0.9)*jupiter_mass
Wispit_mass_lower = ufloat(4.9, 0.6)*jupiter_mass

print("WISPIT 2 planet mass = ", round(Wispit_mass_upper.n,4), " + ",round(Wispit_mass_upper.s,4), " - ", round(Wispit_mass_lower.s,4), "Solar masses")

WISPIT 2 planet mass =  0.0047  +  0.0009  -  0.0006 Solar masses


In [7]:
#### Ok, but we want 2, 5 and 10 jupiter masses, so

mass_upper = jupiter_mass*10
mass_middle = jupiter_mass*5
mass_lower =  jupiter_mass*2

print(f"Mass upper: {mass_upper} solar masses\nMass middle: {mass_middle} solar masses\nMass lower: {mass_lower} solar masses\n----")
print("BUT: the central star is heavier than the sun. Accounting for that, and using a stellar mass of 1.08+0.06-0.17 solar masses")
wispit_star_mass = ufloat(1.08, 0.17) # in solar masses

mass_upper_stellar_mass = mass_upper/wispit_star_mass
mass_middle_stellar_mass = mass_middle/wispit_star_mass
mass_lower_stellar_mass = mass_lower/ wispit_star_mass

print(f"Mass upper: {mass_upper_stellar_mass} Wispit star masses\nMass middle: {mass_middle_stellar_mass} Wipsit star masses\nMass lower: {mass_lower_stellar_mass} Wispit star masses\n----")



Mass upper: 0.0095479190+/-0.0000000020 solar masses
Mass middle: 0.0047739595+/-0.0000000010 solar masses
Mass lower: 0.0019095838+/-0.0000000004 solar masses
----
BUT: the central star is heavier than the sun. Accounting for that, and using a stellar mass of 1.08+0.06-0.17 solar masses
Mass upper: 0.0088+/-0.0014 Wispit star masses
Mass middle: 0.0044+/-0.0007 Wipsit star masses
Mass lower: 0.00177+/-0.00028 Wispit star masses
----


In [8]:
20000*0.05

1000.0